In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from datetime import datetime
import json
import unicodedata
import re

BASE_URL = "https://freebeacon.com"
START_URL = "https://freebeacon.com/"
AJAX_URL = f"{BASE_URL}/wp-admin/admin-ajax.php"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": START_URL,
    "X-Requested-With": "XMLHttpRequest"
}

TARGET_COUNT = 200
MIN_WORDS = 150

#----------------------------------------------------
# Utils
#----------------------------------------------------

def normalize_text_for_nlp(text):

    if not text:
        return ""

    # 1. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace non-breaking space
    text = text.replace("\u00A0", " ")

    # 3. Remove zero-width chars
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)

    # 4. Fix UTF-8 -> Windows-1252 mojibake
    text = (
        text.replace("â€˜", "'")
            .replace("â€™", "'")
            .replace("â€œ", '"')
            .replace("â€", '"')
            .replace("â€“", "-")
            .replace("â€”", "-")
            .replace("â€¦", "...")
    )

    # 5. Remove stray control chars & mojibake remnants
    text = re.sub(r"[\u0000-\u001F\u007F]", " ", text)
    text = re.sub(r"[\u0080-\u009F]", "", text)

    # 6. Normalize punctuation
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
            .replace("–", "-")
            .replace("—", "-")
    )

    # 7. Normalize whitespace
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    
    return text.strip()

# ---------------------------------------------------
# Detect valid article URL
# ---------------------------------------------------
def is_article_url(url):
    parsed = urlparse(url)

    if parsed.netloc != "freebeacon.com":
        return False

    if "/./" in parsed.path:
        return False

    segments = [s for s in parsed.path.strip("/").split("/") if s]

    # Must be category/article-slug
    if len(segments) < 2:
        return False

    if segments[0] in ["tag", "category", "author", "type"]:
        return False
    
    return True

# ---------------------------------------------------
# Extract URLs from HTML
# ---------------------------------------------------
def extract_urls(html, exclude_patterns):
    soup = BeautifulSoup(html, "lxml")
    urls = set()

    for a in soup.find_all("a", href=True):
        full_url = urljoin(BASE_URL, a["href"])

        if not full_url.startswith(BASE_URL):
            continue

        if any(p in full_url for p in exclude_patterns):
            continue

        if is_article_url(full_url):
            urls.add(full_url)

    return urls


# ---------------------------------------------------
# Extract post IDs (for AJAX exclude)
# ---------------------------------------------------
def extract_post_ids(html):
    soup = BeautifulSoup(html, "lxml")
    ids = set()

    for article in soup.find_all("article"):
        article_id = article.get("id")
        if article_id and article_id.startswith("post-"):
            ids.add(article_id.replace("post-", ""))

    return ids

# ---------------------------------------------------
# Extract article data
# ---------------------------------------------------
def extract_article_data(session, url):

    print(f"   → Visiting article: {url}")

    try:
        r = session.get(url, headers=HEADERS, timeout=15)
    except Exception as e:
        print(f"     ! Request failed: {e}")
        return None

    if r.status_code != 200:
        print("     ! Failed to load")
        return None

    soup = BeautifulSoup(r.text, "lxml")

    # -------- TITLE (h1 OR h2) --------
    title_tag = soup.find("h1")
    if not title_tag:
        title_tag = soup.find("h2")

    title = title_tag.get_text(strip=True) if title_tag else None

    if not title:
        print("     ! No title found")
        return None

    # -------- DATE --------
    date = None
    
    # 1️⃣ Try OpenGraph / meta first (most reliable)
    meta_date = soup.find("meta", property="article:published_time")
    if meta_date and meta_date.get("content"):
        try:
            parsed_meta = datetime.fromisoformat(meta_date["content"].replace("Z", "+00:00"))
            date = parsed_meta.date().isoformat()
        except Exception:
            date = normalize_text_for_nlp(meta_date["content"])
    
    # 2️⃣ Fallback: author-date block
    if not date:
        author_block = soup.find("div", class_="author-date")
        if author_block:
            date_divs = author_block.find_all("div")
            if len(date_divs) >= 2:
                raw_date = date_divs[1].get_text(strip=True)
                raw_date = normalize_text_for_nlp(raw_date)
    
                try:
                    parsed_date = datetime.strptime(raw_date, "%B %d, %Y")
                    date = parsed_date.date().isoformat()
                except Exception:
                    date = raw_date

    # -------- BODY --------
    content_div = soup.find("div", class_="article-content")

    if not content_div:
        print("     ! No content div found")
        return None

    # Remove scripts/styles
    for tag in content_div.find_all(["script", "style"]):
        tag.decompose()

    # Remove ads
    for ad in content_div.find_all("div", class_=["in-article-ad-wrapper"]):
        ad.decompose()

    # Remove tag list
    tag_list = content_div.find("div", class_="tag-list")
    if tag_list:
        tag_list.decompose()

    paragraphs = content_div.find_all("p")

    clean_paragraphs = []
    
    for p in paragraphs:
    
        # Remove unwanted nested elements if needed
        for unwanted in p.find_all(["script", "style"]):
            unwanted.decompose()
    
        text = p.get_text(separator=" ", strip=True)
    
        # Normalize whitespace
        text = " ".join(text.split())
    
        if not text:
            continue
    
        if len(text.split()) <= 3:
            continue
    
        clean_paragraphs.append(text)
    
    body_text = "\n\n".join(clean_paragraphs)
    body_text = normalize_text_for_nlp(body_text)
    title = normalize_text_for_nlp(title)

    # -------- WORD COUNT FILTER --------
    word_count = len(body_text.split())

    if word_count < MIN_WORDS:
        print(f"     ✗ Skipped (only {word_count} words)")
        return None

    # -------- ARTICLE ID --------
    slug = urlparse(url).path.strip("/").split("/")[-1]
    article_id = slug

    # -------- TIMESTAMP --------
    crawl_timestamp = datetime.utcnow().isoformat()

    print(f"     ✓ Extracted ({word_count} words)")

    return {
        "title": title,
        "date_published": date,
        "main_body": body_text,
        "url": url,
        "article_id": article_id,
        "word_count": word_count,
        "outlet": "The Washington Free Beacon",
        "label": "right",
        "crawl_timestamp": crawl_timestamp,
    }

# ---------------------------------------------------
# Main crawler
# ---------------------------------------------------
def crawl_articles(exclude_patterns, max_pages=500):

    session = requests.Session()
    all_urls = set()
    exclude_ids = set()
    articles = []

    # 1️⃣ Initial page
    r = session.get(START_URL, headers=HEADERS)
    if r.status_code != 200:
        print("Failed to load start page.")
        return []
        
    all_urls.update(extract_urls(r.text, exclude_patterns))
    exclude_ids.update(extract_post_ids(r.text))

    page = 2

    # 2️⃣ AJAX pagination
    while page <= max_pages and len(all_urls) < TARGET_COUNT:
        payload = {
            "action": "wfb_load_more",
            "page": page,
            "postType": "post",
            "tag": None,
            "category": None,
            "exclude": "[" + ",".join(exclude_ids) + "]"
        }

        response = session.post(AJAX_URL, headers=HEADERS, data=payload)

        if response.status_code != 200:
            break

        try:
            json_data = response.json()
        except Exception as e:
            print(f"AJAX JSON parse failed: {e}")
            break

        html_fragment = json_data.get("data")
        if not html_fragment:
            break

        all_urls.update(extract_urls(html_fragment, exclude_patterns))
        exclude_ids.update(extract_post_ids(html_fragment))

        page += 1

    # 3️⃣ Visit each article
    print(f"Visiting {min(len(all_urls), TARGET_COUNT)} articles...")

    for url in sorted(all_urls):
    
        if len(articles) >= TARGET_COUNT:
            break
    
        data = extract_article_data(session, url)
        if data:
            articles.append(data)

    return articles

# ---------------------------------------------------
# Run
# ---------------------------------------------------
if __name__ == "__main__":

    OUTPUT_FILE = "freebeacon_articles.json"

    print("\n========== STARTING CRAWL ==========\n")

    results = crawl_articles(
        exclude_patterns=[
            "/about/", "/masthead/", "/terms-of-use/",
            "/privacy-policy/", "/./", "/author/",
            "/type/video/", "/newsletters/",
            "/men-of-the-year/", "/arts-culture-opinion/",
            "/fact-check/", "/stiles-section/",
            "/satire/", "/enemies-of-freedom/",
        ]
    )

    print(f"\n========== CRAWL COMPLETE ==========")
    print(f"Valid articles extracted: {len(results)}")

    if not results:
        print("No articles extracted. Exiting.")
        exit()

    # ----------------------------------
    # Dump full dataset to JSON
    # ----------------------------------
    print(f"\nWriting results to {OUTPUT_FILE}...")

    try:
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(
                results,
                f,
                indent=4,
                ensure_ascii=False
            )

        print("✓ JSON file written successfully (UTF-8 encoded).")

    except Exception as e:
        print(f"✗ Failed to write JSON file: {e}")
        exit()

    # ----------------------------------
    # Summary stats
    # ----------------------------------
    total_words = sum(a["word_count"] for a in results)
    avg_words = total_words / len(results)

    print("\n========== SUMMARY ==========")
    print(f"Total Articles: {len(results)}")
    print(f"Total Words: {total_words}")
    print(f"Average Words per Article: {int(avg_words)}")
    print("=================================\n")


========== STARTING CRAWL ==========

Visiting 200 articles...
   → Visiting article: https://freebeacon.com/america/federal-judge-authorizes-evictions-at-condominium-besieged-by-sprawling-homeless-encampment/
     ✓ Extracted (719 words)
   → Visiting article: https://freebeacon.com/america/for-america-at-250-some-jewish-wisdom-on-how-to-last-3000-years/
     ✓ Extracted (705 words)
   → Visiting article: https://freebeacon.com/america/in-sxsw-appearance-columbia-encampment-organizer-mahmoud-khalil-says-its-very-racist-to-ask-him-to-condemn-hamas/
     ✓ Extracted (858 words)
   → Visiting article: https://freebeacon.com/america/mamdani-dimon-offer-clashing-visions-of-new-york-city-and-america-at-250/
     ✓ Extracted (1306 words)
   → Visiting article: https://freebeacon.com/america/nyc-congestion-pricing-scheme-prioritizes-environmental-justice-communities-pumps-money-into-minority-neighborhoods-in-what-could-be-illegal-discrimination/
     ✓ Extracted (1074 words)
   → Visiting ar